# Position Triangulation With Three Receivers

Purpose: prototype the new idea of estimating a person's room position from three receiver/range constraints after a person has already been detected.

Run path: from the repo root, start Jupyter with `uv run jupyter lab` or execute this notebook in an editor using the repo environment.

Fixture / simulated source: this notebook uses a deterministic synthetic room with three surveyed receiver positions and noisy distance estimates. It does not claim current CSI can directly produce metric ranges without calibration.

Expected interpretation: with three reasonably independent receivers, least-squares trilateration should recover a 2D position close to the true point. Error grows when range estimates are noisy or receiver geometry is poor.

Limitations: the notebook demonstrates the position solve only. A real deployment still needs a calibrated conversion from CSI features, ToA, RSSI, UWB, or fingerprint matches into per-receiver distance/range constraints.

In [ ]:
from __future__ import annotations

# Data source option: synthetic fixture or local ESP32 CSI recording.
USE_RECORDING = False
RECORDING_CSV = None  # Set to a specific *_csi.csv path, or leave None to auto-pick from data/recordings.
_RECORDING_MAX_FRAMES = None

from pathlib import Path
import numpy as np

try:
    from ruview.hardware.esp32_capture_analysis import load_esp32_capture
except ImportError:  # Allows the notebook to render in environments without the package installed yet.
    load_esp32_capture = None


def _find_repo_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'ruview').exists():
            return candidate
    return current


def _pick_recording_csv(repo_root, override=None):
    if override:
        return Path(override).expanduser().resolve()
    candidates = sorted((repo_root / 'data' / 'recordings').glob('**/*_csi.csv'))
    return candidates[0] if candidates else None


def _elapsed_seconds_for_capture(capture):
    real_ts = np.asarray(capture.timestamps, dtype=np.float64)
    if np.isfinite(real_ts).sum() >= 2 and float(np.nanmax(real_ts) - np.nanmin(real_ts)) > 0:
        return real_ts - float(np.nanmin(real_ts))
    mono = np.asarray(capture.host_monotonic_ns, dtype=np.float64)
    if mono.size == 0:
        return np.array([], dtype=np.float64)
    return (mono - float(mono[0])) / 1_000_000_000.0


def _fill_invalid(values, mask):
    filled = np.asarray(values, dtype=np.float64).copy()
    filled[~mask] = np.nan
    valid = np.isfinite(filled)
    counts = valid.sum(axis=0)
    sums = np.where(valid, filled, 0.0).sum(axis=0)
    col_means = np.divide(sums, counts, out=np.zeros_like(sums), where=counts > 0)
    rows, cols = np.where(~valid)
    filled[rows, cols] = col_means[cols]
    return filled


def _capture_to_csi_tensor(capture, max_frames=None):
    take = slice(None) if max_frames is None else slice(0, max_frames)
    amp = _fill_invalid(capture.amplitude[take], capture.valid_mask[take])
    phase = _fill_invalid(capture.phase[take], capture.valid_mask[take])
    time_s = _elapsed_seconds_for_capture(capture)[take]
    csi = (amp * np.exp(1j * phase))[:, None, :]
    return csi, time_s


def _effective_sample_rate(time_s):
    time_s = np.asarray(time_s, dtype=np.float64)
    duration = float(time_s[-1] - time_s[0]) if time_s.size >= 2 else 0.0
    return float((time_s.size - 1) / duration) if duration > 0 else 1.0


_recording_repo_root = _find_repo_root()
_recording_path = _pick_recording_csv(_recording_repo_root, RECORDING_CSV)
_recording_capture = None
_recording_csi = None
_recording_time_s = None

if USE_RECORDING:
    if load_esp32_capture is None:
        raise ImportError('ruview.hardware.esp32_capture_analysis.load_esp32_capture is required for recordings')
    if _recording_path is None:
        raise FileNotFoundError('No *_csi.csv recording found under data/recordings; set RECORDING_CSV explicitly.')
    _recording_capture = load_esp32_capture(_recording_path)
    _recording_csi, _recording_time_s = _capture_to_csi_tensor(_recording_capture, _RECORDING_MAX_FRAMES)
    print(f'Using recording: {_recording_path} ({_recording_csi.shape[0]} frames, {_recording_csi.shape[2]} subcarriers)')
else:
    print('Using synthetic fixture. Set USE_RECORDING=True to plot from a local *_csi.csv recording.')


import math

import matplotlib.pyplot as plt
import numpy as np

try:
    from ruview.mat.localization import (
        DistanceEstimate,
        RangeConstraint,
        RangeConstraintFuser,
        SensorPosition,
        rssi_to_distance,
        TriangulationConfig,
        Triangulator,
        validate_range_constraints,
    )
except Exception as exc:
    raise RuntimeError('Run this notebook from the repo environment, e.g. `uv run jupyter lab`.') from exc

# For live triangulation, set USE_RECORDING=True and provide exactly three receiver CSVs here.
# A single receiver recording is not enough to estimate a 2D position.
RECEIVER_RECORDINGS = {
    # 'rx-a': '/path/to/rx-a_csi.csv',
    # 'rx-b': '/path/to/rx-b_csi.csv',
    # 'rx-c': '/path/to/rx-c_csi.csv',
}


In [ ]:
receivers = [
    SensorPosition('rx-a', 0.0, 0.0, 1.2),
    SensorPosition('rx-b', 5.5, 0.0, 1.2),
    SensorPosition('rx-c', 2.4, 4.2, 1.2),
]

# Person position projected to the room floor. The z value is fixed for the 2D solver.
truth_xy = np.array([2.2, 1.7], dtype=np.float64)
range_noise_m = np.array([0.06, -0.04, 0.03], dtype=np.float64)


def receiver_recording_distances(receiver_recordings):
    if not USE_RECORDING:
        return None
    if len(receiver_recordings) != len(receivers):
        raise ValueError('Live triangulation needs one *_csi.csv recording per receiver id in RECEIVER_RECORDINGS.')
    live_distances = []
    for receiver in receivers:
        path = Path(receiver_recordings[receiver.id]).expanduser().resolve()
        capture = load_esp32_capture(path)
        rssi = np.asarray(capture.rssi, dtype=float)
        finite = rssi[np.isfinite(rssi)]
        if finite.size == 0:
            raise ValueError(f'{receiver.id} recording has no finite RSSI values: {path}')
        distance_m = rssi_to_distance(float(np.median(finite)))
        live_distances.append(
            DistanceEstimate(receiver.id, distance_m, confidence=0.55, uncertainty_m=max(0.35 * distance_m, 0.50))
        )
    return live_distances


distances = receiver_recording_distances(RECEIVER_RECORDINGS)
if distances is None:
    distances = []
    for receiver, noise in zip(receivers, range_noise_m):
        receiver_xy = np.array(receiver.xy, dtype=np.float64)
        measured = float(np.linalg.norm(truth_xy - receiver_xy) + noise)
        distances.append(DistanceEstimate(receiver.id, measured, confidence=0.9, uncertainty_m=0.20))

triangulator = Triangulator(TriangulationConfig(max_uncertainty_m=30.0 if USE_RECORDING else 2.0, weighted=True))
estimate = triangulator.trilaterate(receivers, distances, z=1.2)

print('Measured receiver ranges:')
for item in distances:
    print(f'  {item.sensor_id}: {item.distance_m:.2f} m')

if estimate is None:
    raise RuntimeError('Triangulation did not produce an estimate')

print(f'Estimated position: ({estimate.x:.2f}, {estimate.y:.2f}, {estimate.z:.2f}) m')
if USE_RECORDING:
    print('True position:      unavailable for recording-backed mode')
else:
    error_m = float(np.linalg.norm(np.array([estimate.x, estimate.y]) - truth_xy))
    print(f'True position:      ({truth_xy[0]:.2f}, {truth_xy[1]:.2f}, 1.20) m')
    print(f'Horizontal error:   {error_m:.2f} m')
print(f'Uncertainty:        {estimate.uncertainty.horizontal_error:.2f} m')


In [ ]:
constraints = [
    RangeConstraint(item.sensor_id, receiver.xyz, item.distance_m, uncertainty_m=item.uncertainty_m or 0.20)
    for receiver, item in zip(receivers, distances)
]

validation = validate_range_constraints(estimate, constraints, gate_sigma=3.0)
refined = RangeConstraintFuser().refine(estimate, constraints)

print('Range validation:')
print(f'  consistent={validation.consistent}')
print(f'  admitted={validation.admitted_anchor_ids}')
print(f'  rejected={validation.rejected_anchor_ids}')
print(f'  rms residual sigma={validation.rms_residual_sigma:.2f}')
print('\nRefined position:')
print(f'  ({refined.position[0]:.2f}, {refined.position[1]:.2f}, {refined.position[2]:.2f}) m')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)

theta = np.linspace(0.0, 2.0 * math.pi, 360)
for receiver, item in zip(receivers, distances):
    x, y = receiver.xy
    ax.scatter([x], [y], s=90, marker='^', label=receiver.id)
    ax.text(x + 0.08, y + 0.08, receiver.id)
    ax.plot(x + item.distance_m * np.cos(theta), y + item.distance_m * np.sin(theta), linewidth=1, alpha=0.35)

if distances is not None and not USE_RECORDING:
    ax.scatter([truth_xy[0]], [truth_xy[1]], s=90, marker='x', color='black', label='true person')
ax.scatter([estimate.x], [estimate.y], s=90, marker='o', color='tab:red', label='triangulated')
ax.scatter([refined.position[0]], [refined.position[1]], s=60, marker='s', color='tab:purple', label='range-refined')
ax.set_aspect('equal', adjustable='box')
ax.set_xlim(-0.5, 6.0)
ax.set_ylim(-0.5, 5.0)
ax.set_xlabel('room x (m)')
ax.set_ylabel('room y (m)')
ax.set_title('Three-receiver range triangulation')
ax.legend(loc='upper right');

In [ ]:
def estimate_error_for_noise(noise_std_m: float, trials: int = 300, seed: int = 17):
    rng = np.random.default_rng(seed)
    errors = []
    for _ in range(trials):
        noisy_distances = []
        for receiver in receivers:
            base = np.linalg.norm(truth_xy - np.array(receiver.xy, dtype=np.float64))
            noisy_distances.append(DistanceEstimate(receiver.id, float(base + rng.normal(0.0, noise_std_m)), uncertainty_m=max(noise_std_m, 0.05)))
        result = triangulator.trilaterate(receivers, noisy_distances, z=1.2)
        if result is not None:
            errors.append(float(np.linalg.norm(np.array([result.x, result.y]) - truth_xy)))
    return np.asarray(errors, dtype=np.float64)


noise_levels = np.array([0.05, 0.10, 0.20, 0.35, 0.50], dtype=np.float64)
error_sets = [estimate_error_for_noise(level) for level in noise_levels]

for level, errors in zip(noise_levels, error_sets):
    print(f'noise={level:.2f} m -> median error={np.median(errors):.2f} m, p90={np.percentile(errors, 90):.2f} m')

fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
ax.boxplot(error_sets, labels=[f'{level:.2f}' for level in noise_levels], showfliers=False)
ax.set_xlabel('range noise std (m)')
ax.set_ylabel('position error (m)')
ax.set_title('Triangulation sensitivity to range noise');

## How This Connects To CSI

The triangulation stage needs one distance-like constraint per receiver. In a real RuView setup, possible inputs are calibrated RSSI/path-loss distances, time-of-arrival, UWB/mmWave ranges, or learned CSI fingerprints that emit a range constraint with uncertainty.

The next research step is not the least-squares solve itself; it is learning or calibrating the mapping from CSI features to stable per-receiver range constraints.